Zadaniem klasyfikacyjnym dla modeli było ustalenie liczby "1" nad przekątną w postaci Jordana.

Macierze losowe są generowane w następujący sposób:
* stwórz macierz $J$ z odpowiednią liczbą $1$ nad przekątną oraz z wartościami własnymi $\lambda = 1$ na przekątnej
* wylosuj macierz $S$, taką że $\kappa(S) < 10^5$
* $ X = S^{-1} J S$
* macierz jest normalizowana (względem normy Frobeniusa): $ X = 1/\|X\|_F X$

Losowanie macierzy $S$ odbywa się wg różnych metod:
* `random`: $S$ generowana przez `np.random.rand(d, d)`
* `upper`: $S$ jest górnotrójkątna, `S = np.triu(np.random.rand(d, d))`
* `int`: $S$ jest o wyrazach całkowitych z zakresu 1 do 100: `S = np.random.randint(0, 100, size=(d, d))`
* `ortho`: $S$ jest ortonormalna `A = np.random.rand(d,d) \ S, _ = np.linalg.qr(A)` 

Próbowane było generowanie zbioru treningowego i testowego wg różnych metod; rozmiar macierzy $5 \times 5$. 

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from jordanutils import *
import pandas as pd

rng = np.random.default_rng(seed=21)

In [3]:
import torch_directml

torch_directml.device()

device(type='privateuseone', index=0)

In [ ]:
# ---- Define the PyTorch model ----
class SimpleNN(nn.Module):
    def __init__(self, d):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(d * d, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, d),
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x


# ---- Training + Evaluation function ----
def train_and_test_model(train_mode, test_mode, verbose=1, epochs=50):
    # --- Device selection logic ---
    try:
        import torch_directml

        device = torch_directml.device()
        backend = "directml"
    except ImportError:
        if torch.cuda.is_available():
            device = torch.device("cuda")
            backend = "cuda"
        else:
            device = torch.device("cpu")
            backend = "cpu"

    if verbose:
        print(f"Using device: {device} (backend: {backend})")

    # --- Dataset setup ---
    d = 5
    dataset_size = 50000
    manager = LabelsManager([0], [1], [2], [3], [4], dataset_size=dataset_size)
    X, y = generate_testset(d, manager, mode=train_mode)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=21
    )

    # --- Convert to tensors ---
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.long)

    # --- DataLoaders ---
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

    # --- Model, Loss, Optimizer ---
    model = SimpleNN(d).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # --- Early Stopping setup ---
    best_val_acc = 0.0
    patience = 3
    counter = 0
    best_weights = None

    # --- Training Loop ---
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        # --- Validation Loop ---
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                pred_labels = preds.argmax(1)
                correct += (pred_labels == yb).sum().item()
                total += yb.size(0)
        val_acc = correct / total

        if verbose:
            print(f"Epoch {epoch+1:02d} - Val Acc: {val_acc:.4f}")

        # --- Early Stopping ---
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            counter = 0
            best_weights = model.state_dict()
        else:
            counter += 1
            if counter >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1}")
                break

    # --- Restore best model ---
    if best_weights is not None:
        model.load_state_dict(best_weights)

    # --- Testing ---
    manager.dataset_size = 1000
    X_test, y_test = generate_testset(d, manager, mode=test_mode)
    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(X_test)
        y_predicted = torch.argmax(outputs, dim=1).cpu().numpy()

    return y_test, y_predicted

In [14]:
modes = ["random", "int", "upper", "lower", "ortho"]

results = pd.DataFrame(columns=modes)
results.index.name = 'Training'
results.columns.name = 'Test'

results_dict = {}

# column: test mode; row: training mode

for train_mode in modes:
    for test_mode in modes: 
        print(f"### Running training mode: {train_mode}, test mode: {test_mode} ###")
        y_test, y_pred = train_and_test_model(train_mode, test_mode, verbose=1, epochs=50)
        results_dict[(train_mode, test_mode)] = (y_test, y_pred)
        accuracy = accuracy_score(y_test, y_pred)
        results.loc[train_mode, test_mode] = accuracy
        

### Running training mode: random, test mode: random ###
Using device: privateuseone:0 (backend: directml)
Epoch 01 - Val Acc: 0.5965
Epoch 02 - Val Acc: 0.6321
Epoch 03 - Val Acc: 0.6503
Epoch 04 - Val Acc: 0.6615
Epoch 05 - Val Acc: 0.6556
Epoch 06 - Val Acc: 0.6615
Epoch 07 - Val Acc: 0.6772
Epoch 08 - Val Acc: 0.6780
Epoch 09 - Val Acc: 0.6783
Epoch 10 - Val Acc: 0.6790
Epoch 11 - Val Acc: 0.6890
Epoch 12 - Val Acc: 0.6825
Epoch 13 - Val Acc: 0.6892
Epoch 14 - Val Acc: 0.6873
Epoch 15 - Val Acc: 0.6924
Epoch 16 - Val Acc: 0.6925
Epoch 17 - Val Acc: 0.6967
Epoch 18 - Val Acc: 0.6917
Epoch 19 - Val Acc: 0.6957
Epoch 20 - Val Acc: 0.6985
Epoch 21 - Val Acc: 0.6969
Epoch 22 - Val Acc: 0.6964
Epoch 23 - Val Acc: 0.6944
Early stopping at epoch 23
### Running training mode: random, test mode: int ###
Using device: privateuseone:0 (backend: directml)
Epoch 01 - Val Acc: 0.6059
Epoch 02 - Val Acc: 0.6403
Epoch 03 - Val Acc: 0.6539
Epoch 04 - Val Acc: 0.6618
Epoch 05 - Val Acc: 0.6687
Epoch 

In [15]:
display(results)

Test,random,int,upper,lower,ortho
Training,,,,,
random,0.6886,0.6898,0.5916,0.654,0.5702
int,0.7112,0.7036,0.5662,0.6466,0.5694
upper,0.4,0.3912,0.9826,0.8124,0.4288
lower,0.4056,0.4044,0.8184,0.985,0.4032
ortho,0.4018,0.4026,0.4218,0.4076,0.992


Widać, że nie ma żadnej różnicy między losowaniem z macierzy przejścia całkowitych czy rzeczywistych. Używanie macierzy przejścia górnotrójkątnych w obu przypadkach pogorszyło rezultaty. 

Model doskonale radzi sobie, gdy zawęzimy problem do sytuacji, gdy wektory własne/główne są do siebie ortogonalne (czego można było się spodziewać). Taką sytuację jednakże równie dobrze rozwiązuje obliczalny numerycznie rozkład Schura.

Z drugiej strony, wytrenowany na losowych danych model, słabo radzi sobie z rozróżnianiem wielkości bloków, gdy $S$ jest ortonormalna. 